# Prompting Claude for "JSON Mode" (TypeScript)

Claude doesn't have a formal "JSON Mode" with constrained sampling. But not to worry — you can still get reliable JSON from Claude! This recipe will show you how using the Anthropic TypeScript SDK.

## 1. Setup

This notebook uses the **Deno kernel** for TypeScript. Deno handles package imports natively — no `npm install` needed.

Set your `ANTHROPIC_API_KEY` in a `.env` file at the project root:
```
ANTHROPIC_API_KEY=sk-ant-...
```

In [3]:
import Anthropic from 'npm:@anthropic-ai/sdk';
import { load } from 'https://deno.land/std@0.220.0/dotenv/mod.ts';

// Load .env from project root (one level up from misc/) into Deno.env
await load({ envPath: '../.env', export: true });

const apiKey = Deno.env.get('ANTHROPIC_API_KEY');
if (!apiKey) throw new Error('ANTHROPIC_API_KEY not found in .env');

const client = new Anthropic({ apiKey });
const MODEL_NAME = 'claude-sonnet-4-6';
console.log('Anthropic client initialized. Model:', MODEL_NAME);

Anthropic client initialized. Model: claude-sonnet-4-6


## 2. Default Behavior: Claude's JSON Output

First, let's look at Claude's default behavior when asked for JSON — it typically adds a preamble before the actual data.

In [ ]:
const response = await client.messages.create({
  model: MODEL_NAME,
  max_tokens: 1024,
  messages: [
    {
      role: 'user' as const,
      content: 'Give me a JSON dict with names of famous athletes & their sports.',
    },
  ],
});

const message = response.content[0].type === 'text' ? response.content[0].text : '';
console.log(message);

## 3. Extract JSON from Response

Claude followed instructions and returned a dictionary. We can extract it by finding the first `{` and last `}` in the response string.

In [ ]:
const extractJson = (response: string): Record<string, unknown> => {
  const jsonStart = response.indexOf('{');
  const jsonEnd = response.lastIndexOf('}');
  if (jsonStart === -1 || jsonEnd === -1) throw new Error('No JSON object found in response');
  return JSON.parse(response.slice(jsonStart, jsonEnd + 1)) as Record<string, unknown>;
};

const parsed = extractJson(message);
console.log(JSON.stringify(parsed, null, 2));

## 4. System Prompt for Direct JSON Output

What if we want Claude to skip the preamble entirely? We can use a **system prompt** to instruct Claude to output only raw JSON — no explanation, no preamble, no markdown code fences.

> **Note:** Assistant message prefill (ending the conversation with an `assistant` message) is no longer supported in current Claude models. The system prompt approach achieves the same result.

In [ ]:
const prefillResponse = await client.messages.create({
  model: MODEL_NAME,
  max_tokens: 1024,
  system: 'Output only raw JSON. No explanation, no preamble, no markdown code fences.',
  messages: [
    {
      role: 'user' as const,
      content: 'Give me a JSON dict with names of famous athletes & their sports.',
    },
  ],
});

const prefillMessage = prefillResponse.content[0].type === 'text' ? prefillResponse.content[0].text : '';
console.log(prefillMessage);

## 5. Parse Direct JSON Response

Since Claude now outputs raw JSON directly (no preamble, no code fences), we can parse it immediately without any string manipulation.

In [ ]:
const outputJson = JSON.parse(prefillMessage) as Record<string, string>;
console.log(JSON.stringify(outputJson, null, 2));

## 6. Using XML Tags for Complex Multi-JSON Prompts

For prompts that produce **multiple JSON objects**, a simple `{`/`}` string search won't work reliably. Instead, instruct Claude to wrap each JSON in named XML tags so we can extract them precisely.

In [ ]:
const multiJsonResponse = await client.messages.create({
  model: MODEL_NAME,
  max_tokens: 1024,
  messages: [
    {
      role: 'user' as const,
      content: `Give me a JSON dict with the names of 5 famous athletes & their sports.
Put this dictionary in <athlete_sports> tags.

Then, for each athlete, output an additional JSON dictionary. In each of these additional dictionaries:
- Include two keys: the athlete's first name and last name.
- For the values, list three words that start with the same letter as that name.
Put each of these additional dictionaries in separate <athlete_name> tags.`,
    },
    {
      role: 'user' as const,
      content: 'Here is the JSON requested:',
    },
  ],
});

const multiMessage = multiJsonResponse.content[0].type === 'text' ? multiJsonResponse.content[0].text : '';
console.log(multiMessage);

## 7. Extract JSON from XML Tags

Now we can use a regex helper to pull each tagged block and parse it independently.

In [ ]:
function extractBetweenTags(tag: string, text: string, strip = false): string[] {
  const regex = new RegExp(`<${tag}>([\\s\\S]+?)<\\/${tag}>`, 'g');
  const results: string[] = [];
  let match: RegExpExecArray | null;
  while ((match = regex.exec(text)) !== null) {
    results.push(strip ? match[1].trim() : match[1]);
  }
  return results;
}

// Extract and parse the single athlete_sports dict
// Use extractJson (defined in cell 6) to strip markdown fences before parsing
const athleteSportsDict = extractJson(extractBetweenTags('athlete_sports', multiMessage, true)[0]) as Record<string, string>;
console.log('=== Athlete Sports ===');
console.log(JSON.stringify(athleteSportsDict, null, 2));

// Extract and parse each individual athlete_name dict
const athleteNameDicts = extractBetweenTags('athlete_name', multiMessage, true).map((d: string) => extractJson(d));
console.log('\n=== Athlete Name Dicts ===');
console.log(JSON.stringify(athleteNameDicts, null, 2));

## Summary

So to recap:

- You can use **string parsing** to extract text between ` ```json ` and ` ``` ` fences to get JSON.
- You can **remove preambles** via a **system prompt** instructing Claude to output raw JSON only.
- You can use a **stop sequence** to cut off text after the JSON closes.
- You can instruct Claude to **wrap JSON in XML tags** to cleanly collect multiple JSON objects from a complex response.